<a href="https://colab.research.google.com/github/BraedynL0530/CanopyWatch/blob/master/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#network stuff here when i leave collab

In [ ]:
import io
import requests
import rasterio
import numpy as np

#DO NOT RUN LOCALLY MY PC WILL EXPLODE!!!!!!!!!!
fastapi_url = "http://127.0.0.1:8000/api/get-satellite-patch" #wont work in google collab(unless i tunnell) this is for deploy/temp link
print("Fetching data from server...")
response = requests.get(fastapi_url)

if response.status_code == 204:
    print("No large NVDI change")
    exit()
elif response.status_code != 200:
    print(f"Error fetching: {response.status_code}")
    exit()

tiff_bytes = response.content

with rasterio.open(io.BytesIO(tiff_bytes)) as src:
    print("--- Raster Metadata ---")
    print(f"Width x height: {src.width}x{src.height}")
    print(f"Band Count: {src.count}")
    print(f"Coordinate System: {src.crs}")
    print(f"Geotransform Matrix:\n{src.transform}")
    print("-----------------------")

    #Shape: (4, 512, 512)  (Bands, Height, Width)
    full_stack = src.read()

    #slice the first 3 bands out for visual evidence / frontend
    rgb_channels = full_stack[0:3, :, :]  # Shape: (3, 512, 512)


    nir_channel = full_stack[3, :, :]     # Shape: (512, 512)

print("!!unpacked channels!!")

In [ ]:
#model
import torch
import torch.nn as nn

class forestClassifier(nn.Module):
    def __init__(self):
        super(forestClassifier, self).__init__()
        self.encoder = nn.Conv2d(4, 64, kernel_size=3,  padding=1)
        self.pool = nn.MaxPool2d(kernal_size=2, stride=2)

        self.bottleneckConv = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.upsample = nn.Conv2d(128,64,kernel_size=2,stride=2)

        self.finalConv = nn.Conv2d(64,1,kernal_size=1)

        self.relu = nn.ReLU()
    def forward(self, x):
        x1 = self.relu(self.encoder(x))
        x2 = self.pool(self.x1)
        b=self.relu(self.bottleneckConv(x2))
        u=self.relu(self.upsample(b))
        out = self.finalConv(u)
        return out



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

class DeforestationDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        img_np = np.array(item['image'], dtype=np.float32) / 255.0
        mask_np = np.array(item['mask'], dtype=np.float32) / 255.0

        if len(img_np.shape) == 3:
            img_np = np.transpose(img_np, (2, 0, 1))

        if img_np.shape[0] == 3:
            nir_pad = np.zeros((1, img_np.shape[1], img_np.shape[2]), dtype=np.float32)
            img_np = np.concatenate((img_np, nir_pad), axis=0)

        if len(mask_np.shape) == 2:
            mask_np = np.expand_dims(mask_np, axis=0)

        return torch.tensor(img_np), torch.tensor(mask_np)


def train():
    print("1. Downloading/Loading Kaggle Dataset via Hugging Face...")
    hf_dataset = kagglehub.load_dataset(
        KaggleDatasetAdapter.HUGGING_FACE,
        "akhilchibber/deforestation-detection-dataset",
        ""
    )

    train_data = hf_dataset['train'] if 'train' in hf_dataset else hf_dataset

    print("2. Preparing DataLoaders...")
    pytorch_dataset = DeforestationDataset(train_data)
    dataloader = DataLoader(pytorch_dataset, batch_size=8, shuffle=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ForestClassifier().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 10
    print(f"3. Starting training on {device}...")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (images, masks) in enumerate(dataloader):
            images = images.to(device)
            masks = masks.to(device)

            predictions = model(images)

            loss = criterion(predictions, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | Loss: {loss.item():.4f}")

        #end of Epoch Summary
        avg_loss = epoch_loss / len(dataloader)
        print(f"--- Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f} ---")

    print("saving weights")
    torch.save(model.state_dict(), 'canopyguard_v1.pth')

if __name__ == "__main__":
    train()


In [ ]:
#porbally more netowrok styuff + infrence with model/
#